# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content piece**, measures over a trailing 90-day window. All metrics are aggregated to the single 90-day snapshot per piece.

**Time window:** Trailing 90 days (all 30k rows share the same end date)

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/reezcon/First-ML-Pipeline/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("CHECK: Is one row = one content piece?")
print(f"Total rows: {len(df):,}")
print(f"Distinct content_ids: {df["content_id"].nunique():,}")
print(f"Duplicates: {len(df) - df["content_id"].nunique()}") #should be 0

grain_check = df.groupby("content_id").size()
if (grain_check == 1).all():
  print("Grain holds: Each content_id appears exactly once")
else:
  print("Grain broken: Some content_ids repeat")
  print(grain_check[grain_check > 1].head())

CHECK: Is one row = one content piece?
Total rows: 30,000
Distinct content_ids: 30,000
Duplicates: 0
Grain holds: Each content_id appears exactly once


In [ ]:
print("All columns in the started dataset:")
for i, col in enumerate (df.columns, 1):
  dtype = df[col].dtype
  print(f"{i:2d}. {col:30s} | {str(dtype):15s}")

# Show shapes of features vs label
print("FEATURE SELECTION SUMMARY")

features = ["content_type", "position_tier", "freshness_tier", "word_count", "competition_level", "cpc", "search_volume"]
label = ["engagement_rate"]
context = ["content_id", "client_id"]

print(f"\nFeatures ({len(features)}):")
for f in features:
  print(f"   -{f}: {df[f].dtype}")

print(f"\nLabel ({len(label)}):")
for l in label:
  print(f"   -{l}: {df[l].dtype}")

print(f"\nContext ({len(context)}):")
for c in context:
  print(f"   -{c}: {df[c].dtype}")


All columns in the started dataset:
 1. content_id                     | object         
 2. client_id                      | object         
 3. search_volume                  | float64        
 4. competition                    | float64        
 5. competition_level              | object         
 6. cpc                            | float64        
 7. content_type                   | object         
 8. main_intent                    | object         
 9. word_count                     | float64        
10. char_count                     | float64        
11. provider_used                  | object         
12. model_used                     | object         
13. impressions_90d                | int64          
14. clicks_90d                     | int64          
15. pageviews_90d                  | int64          
16. sessions_90d                   | int64          
17. users_90d                      | int64          
18. engaged_sessions_90d           | int64          
19. ai_ses

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:**

*   content_type: categories, such as keyword article, feedly article, and comparison article
*   position_tier: categories, such as top_3, page_1, striking, page_3_5, deep, and no_data
*   freshness_tier: categories, such as 0-30, 31-90, 91-180, and 181+ days since last update
*   word_count: numeric, word count of the article
*   competition_level: categories, such as LOW, MEDIUM, HIGH
*   cpc: numeric, cost-per-click for the target keyword
*   search_volume: numeric, estimated search volume for keyword

**Label:**

*   engagement_rate: numeric (0-100) calculated by engaged_sessions_90d / sessions_90d * 100
This is the outcome we want to predict

**Context:** (for grouping, joining, and splits, not for the model)

*   content_id: pseudonymous content identifier
*   client_id: pseudonymous client identifier

**Excluded:**

*   trend_direction: **leakage** directly derived from trend_pct. Using this means we would be using the label to predict the label.
*   trend_pct: **leakage**

















## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Grain: One row = one content piece
print("CHECK: Is one row = one content piece?")
print(f"Total rows: {len(df):,}")
print(f"Distinct content_ids: {df["content_id"].nunique():,}")
print(f"Duplicates: {len(df) - df["content_id"].nunique()}") #should be 0

grain_check = df.groupby("content_id").size()
if (grain_check == 1).all():
  print("Grain holds: Each content_id appears exactly once")
else:
  print("Grain broken: Some content_ids repeat")
  print(grain_check[grain_check > 1].head())

CHECK: Is one row = one content piece?
Total rows: 30,000
Distinct content_ids: 30,000
Duplicates: 0
Grain holds: Each content_id appears exactly once


In [ ]:
# Counts per category

print("Distribution of features:")
print("\ncontent_type:")
print(df["content_type"].value_counts())

print("\nposition_tier:")
print(df["position_tier"].value_counts())

print("\nfreshness_tier:")
print(df["freshness_tier"].value_counts())

print("\ncompetition_level:")
print(df["competition_level"].value_counts(dropna=False))

Distribution of features:

content_type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

position_tier:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

freshness_tier:
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64

competition_level:
competition_level
LOW       22896
HIGH       2658
NaN        2610
MEDIUM     1836
Name: count, dtype: int64


In [ ]:
# Missing values check

print("Missingness check")
print("Missing counts (absolute & %):")
for col in features:
  missing = df[col].isna().sum()
  pct = 100 * missing/len(df)
  print(f"{col:25s}: {missing:6,d} rows ({pct:5.2f}%)")

print("Missingness by content_type:")

for col in ["word_count", "search_volume", "competition_level", "cpc"]:
  print(f"\n{col}:")
  cross = pd.crosstab(df["content_type"], df[col].isna(), margins=True)
  # Rename the columns based on their boolean values
  # False means it has a value, True means it's missing
  new_column_names = {False: "Has value", True: "Missing"}
  cross = cross.rename(columns=new_column_names)
  print(cross)
  print(f"  -> Missing heavily concentrated? Check if content_type -> missingness.")

Missingness check
Missing counts (absolute & %):
content_type             :      0 rows ( 0.00%)
position_tier            :      0 rows ( 0.00%)
freshness_tier           :      0 rows ( 0.00%)
word_count               :  7,699 rows (25.66%)
competition_level        :  2,610 rows ( 8.70%)
cpc                      :  2,468 rows ( 8.23%)
search_volume            :  2,468 rows ( 8.23%)
Missingness by content_type:

word_count:
word_count          Has value  Missing    All
content_type                                 
comparison article        697        0    697
feedly article           2096        0   2096
keyword article         19508     7699  27207
All                     22301     7699  30000
  -> Missing heavily concentrated? Check if content_type -> missingness.

search_volume:
search_volume       Has value  Missing    All
content_type                                 
comparison article        697        0    697
feedly article              0     2096   2096
keyword article         

In [ ]:
# Label distribution

print("Label Distribution (engagement_rate):")
print(df["engagement_rate"].describe())
print(f"Zeros (no engagement): {(df["engagement_rate"] == 0).sum():,} rows ({100 * (df["engagement_rate"] == 0).sum()/len(df):.1f}%)")
print(f"Min (non-zero): {df[df["engagement_rate"] > 0]["engagement_rate"].min():.2f}")
print(f"Max: {df["engagement_rate"].max():.2f}")

Label Distribution (engagement_rate):
count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64
Zeros (no engagement): 21,629 rows (72.1%)
Min (non-zero): 0.05
Max: 100.00


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limts - what this data cannot tell you:

1.   **No causation:** Correlation only. High position may correlate with engagement, but we cannot prove high ranking causes more engagement or vice versa.
2.   **Snapshot:** This is a singly 90-day aggregate per piece. We cannot observe how signals changed week-to-week.
3. **Engagement depends on traffic:** The model trainlys only on pages with measurable traffic.
4. **Client anonymity:** Insights are directional as we cannot see which strategies worked for which real clients.



## Self-check

Before you submit, confirm each line honestly:

- [-] Every section above is filled — markdown thinking AND the code that backs it
- [-] The notebook runs top to bottom with no errors (Runtime → Run all)
- [-] No client names, URLs, or private queries anywhere
- [-] My claims use careful words: observed, measured, directional, decision-support
- [-] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.